In [ ]:
import os
from dotenv import load_dotenv
from supabase import create_client, Client

In [ ]:
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

In [ ]:
#Testing Data Injection - DO NOT RE RUN
new_patient = {
    "date_of_birth": "1945-05-12",
    "current_city": "Singapore",
    "spouse_name": "Martha",
    "kids_names": "David, Sarah",
    "former_profession": "Engineer",
    "favorite_pet_name": "Buddy",
    "wedding_year": 1970
}

insert_response = supabase.table("patient_details").insert(new_patient).execute()
print("Just added:", insert_response.data)
print("-" * 30)

fetch_response = supabase.table("patient_details").select("*").execute()
print("Everything in the table:", fetch_response.data)

In [ ]:
def fetch_patient_data():
    res = supabase.table("patient_details").select("*").limit(1).execute()
    return res.data[0] if res.data else {}

patient_data = fetch_patient_data()

In [ ]:
patient_data

{'id': '0f5daa09-6a1b-404a-83dd-02bdd12d7243',
 'date_of_birth': '1945-05-12',
 'current_city': 'Singapore',
 'spouse_name': 'Martha',
 'kids_names': 'David, Sarah',
 'former_profession': 'Engineer',
 'childhood_hometown': None,
 'favorite_pet_name': 'Buddy',
 'wedding_year': 1970,
 'created_at': '2026-03-15T06:53:43.874377+00:00'}

In [ ]:
#Testing Data Injection for 7 days - DO NOT RE RUN
from datetime import datetime, timedelta

TEST_PATIENT_ID = "0f5daa09-6a1b-404a-83dd-02bdd12d7243" 

today = datetime.now()
dummy_logs = []

daily_scenarios = [
    {"days_ago": 7, "results": [True, True, False, True, False]},
    {"days_ago": 6, "results": [True, False,True, True, False]},
    {"days_ago": 5, "results": [True, True, False]},     
    {"days_ago": 4, "results": [True, True, False, False, False, True]},                  
    {"days_ago": 3, "results": [False, False, True, False]}, 
    {"days_ago": 2, "results": [False, False]},    
    {"days_ago": 1, "results": [True, False, True]}      
]

facts = ["spouse_name", "childhood_hometown", "favorite_pet_name", "wedding_year", "former_profession"]

for scenario in daily_scenarios:
    target_date = today - timedelta(days=scenario["days_ago"])
    
    for i, is_correct in enumerate(scenario["results"]):
        dummy_logs.append({
            "patient_id": TEST_PATIENT_ID,
            "fact_category": facts[i % len(facts)], 
            "bot_question": "Dummy question?",
            "user_answer": "Dummy answer.",
            "is_correct": is_correct,
            "created_at": target_date.isoformat()
        })

response = supabase.table("assessment_logs").insert(dummy_logs).execute()
print(f"Injected {len(dummy_logs)} test logs across 5 days!")

Injected 28 test logs across 5 days!


**7-Day Cognitive Score Calculation**

In [ ]:
#Cognitive score calculation
def cog_score(patient_id):

    seven_days_ago = (datetime.now() - timedelta(days=7)).isoformat()
    response = supabase.table("assessment_logs") \
        .select("is_correct") \
        .eq("patient_id", patient_id) \
        .gte("created_at", seven_days_ago) \
        .execute()
        
    logs = response.data
    
    if not logs:
        return None 

    total_questions = len(logs)
    correct_answers = sum(1 for log in logs if log["is_correct"])
    
    score_percentage = (correct_answers / total_questions) * 100
    
    return round(score_percentage, 1)

score = cog_score(TEST_PATIENT_ID)
print(f"7-Day Rolling Cognitive Score: {score}%")

7-Day Rolling Cognitive Score: 47.8%
